# Перевірки (заміна pytest)

Ключові інваріанти проєкту як інлайн-асерти. Підтягуємо модулі через `%run`. Кожна клітинка друкує ✅ при успіху.

**Залежності:** `%run` 03_data_prep.ipynb, 06_inference.ipynb

In [ ]:
%run 03_data_prep.ipynb
%run 06_inference.ipynb
import numpy as np, pandas as pd

### Тест 1 — контракт фічей (колонки, winsorize, emoji, циклічність)

In [ ]:
def one(cap='', dur=30.0, t=1_700_000_000):
    return features.engineer_features(pd.DataFrame([{'description':cap,'duration':dur,'create_time':t}])).iloc[0]
assert list(features.engineer_features(pd.DataFrame([{'description':'hi','duration':10,'create_time':1_700_000_000}])).columns) == features.FEATURE_COLUMNS
assert one('x'*5000)['char_len'] == config.CHAR_LEN_CLIP        # winsorize
assert one('I ❤️ this 🔥🔥')['n_emoji'] == 3              # variation selector не рахується окремо
assert -1 <= one(t=1_700_000_000)['hour_sin'] <= 1               # циклічне кодування
r = one('Best!! #a #b @u 🔥'); assert r['n_hashtags']==2 and r['n_mentions']==1 and r['has_exclam']==1
print('✅ Тест 1: контракт фічей')

### Тест 2 — стіна проти витоку (leakage guard)

In [ ]:
X = features.engineer_features(pd.DataFrame([{'description':'hi','duration':10,'create_time':1_700_000_000,'play_count':999}]))
assert 'play_count' not in X.columns                            # post-hoc не потрапляє у фічі
try:
    bad = X.copy(); bad['play_count']=1; features.assert_no_leakage(bad); raise SystemExit('guard не спрацював')
except AssertionError:
    pass
try:
    inference.assert_no_posthoc({'caption':'x','digg_count':5}); raise SystemExit('guard не спрацював')
except ValueError:
    pass
print('✅ Тест 2: leakage guard')

### Тест 3 — мітка заморожена на train

In [ ]:
tr = pd.DataFrame({'author_unique_id':['a','a','a'],'play_count':[100,100,100],
                   'digg_count':[10,20,30],'share_count':[0,0,0],'comment_count':[0,0,0],'collect_count':[0,0,0]})
thr = labels.fit_creator_thresholds(tr); assert round(thr['per_creator']['a'],3)==0.2
te = pd.DataFrame({'author_unique_id':['a','a'],'play_count':[100,100],
                   'digg_count':[25,15],'share_count':[0,0],'comment_count':[0,0],'collect_count':[0,0]})
assert list(labels.make_labels(te, thr)) == [1.0, 0.0]            # пороги з train застосовані до test
print('✅ Тест 3: freeze мітки на train')

### Тест 4 — inference ніколи не падає на брудному вводі

In [ ]:
if not inference.is_trained():
    print('⚠️ пропуск: спершу запусти 05_train.ipynb (немає models/model.joblib)')
else:
    VALID = {'Post','Do not post','Unsure'}
    bad = [dict(), dict(caption=None,duration=None,when=None),
           dict(caption='x'*5000, duration=-10, when=0),
           dict(caption='ok', duration=10**18, when=float('inf')),
           dict(caption='ok', duration=True, when='not-a-date')]
    for kw in bad:
        rr = inference.recommend(**kw); assert rr['recommendation'] in VALID and 0<=rr['probability']<=1
    assert inference.parse_tiktok_url(12345) == {}                # не-рядок не падає
    print('✅ Тест 4: inference robust ({} кейсів)'.format(len(bad)))